In [83]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import fuzz

import spotipy
from spotipy.oauth2 import SpotifyOAuth
import json
import requests






# TODO: what happens when there the song is not in the dataset? (filter rare songs?)
# TODO: have it spit out the artists as well
# TODO: get the API to work (take your top songs from api, run algo for all top songs (steal formatting from Varun))
# TODO: have it take in more than 1 song? prob not (see above)

In [84]:
# CONSTANTS

LEAST_SONGS = 100
NUM_COLS = 1000
FRAC = .1


In [85]:
# read in data
music_info = pd.read_csv("data/Music_Info.csv",
                        sep=',')
user_info = pd.read_csv("data/User Listening History.csv",
                        sep=',')

In [86]:
# merge/clean datasets

# select users who have listened to at least 16 songs FIRST
user_counts = user_info['user_id'].value_counts()
valid_users = user_counts[user_counts >= 16].index

# filter before merge
user_info_filtered = user_info[user_info['user_id'].isin(valid_users)]

# merge smaller
songs = pd.merge(user_info_filtered, music_info, on="track_id", how="left")

# remove duplicates AFTER merge
songs.drop_duplicates(['user_id', 'track_id'], inplace=True)


# select users who have listened to at least 16 songs (number can be changed)
song_per_user = songs.groupby('user_id')['track_id'].count()
song_16_id = song_per_user[song_per_user > LEAST_SONGS].index.to_list()
df_song_id_more_16 = songs[songs['user_id'].isin(song_16_id)].reset_index(drop=True)

In [87]:
# select users who have listened to at least 16 songs (number can be changed)
song_per_user = songs.groupby('user_id')['track_id'].count()
song_16_id = song_per_user[song_per_user > LEAST_SONGS].index.to_list()
df_song_id_more_16 = songs[songs['user_id'].isin(song_16_id)].reset_index(drop=True)

In [124]:
# randomize and grab a part of the dataset
# songs_sample = df_song_id_more_16.sample(frac=FRAC)
songs_sample = df_song_id_more_16
print(len(songs_sample))


473081


In [89]:
print(music_info.columns)


Index(['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id',
       'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='object')


In [ ]:
# convert dataframe to sparce matrix
songs_sample['track_id'] = songs_sample['track_id'].astype('category')
songs_sample['user_id'] = songs_sample['user_id'].astype('category')

df_songs_features = songs_sample.pivot_table(
    index='track_id',
    columns='user_id',
    values='playcount',
    fill_value=0
)

mat_songs_features = csr_matrix(df_songs_features.values)




print(df_songs_features.head())

C:\Users\nflan\AppData\Local\Temp\ipykernel_8604\3785675948.py:5: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_songs_features = songs_sample.pivot_table(


user_id             001889ee41b5f31c404a1abe4af07b2377fa776b  \
track_id                                                       
TRAAAED128E0783FAB                                       0.0   
TRAABJS128F9325C99                                       0.0   
TRAACER128F4290F96                                       0.0   
TRAACIR128F42963AC                                       0.0   
TRAACKM12903CE5BE9                                       0.0   

user_id             001fd039ab4472039d22f9481bb5c5d376c3032f  \
track_id                                                       
TRAAAED128E0783FAB                                       0.0   
TRAABJS128F9325C99                                       0.0   
TRAACER128F4290F96                                       0.0   
TRAACIR128F42963AC                                       0.0   
TRAACKM12903CE5BE9                                       0.0   

user_id             0025bfe6248070545d23721083acd3f60451da4f  \
track_id                              

In [111]:
# list of unique songs
df_unique_songs = music_info.drop_duplicates(subset='track_id').set_index('track_id')

song_metadata = music_info.set_index('track_id').loc[df_songs_features.index][['name', 'artist']]

# gives each song a new unique id
decode_id_song = {
    row['name']: i
    for i, row in enumerate(song_metadata.to_dict('records'))
}

decode_id_artists = {
    i: (row['name'], row['artist'])
    for i, row in enumerate(song_metadata.to_dict('records'))
}

decode_id_title_artist = {
    i: f"{title} - {artist}"
    for i, (title, artist) in decode_id_artists.items()
}

decode_id_song


{"It's About Time": 0,
 'Auburn and Ivory': 1,
 'Setting Fire to Sleeping Giants': 2,
 'Cradle In The Crater': 3,
 'Ionized': 4,
 'Double Feature': 5,
 'Seaweed': 6,
 'Brain Dead': 7,
 'The Beacon': 8,
 'Bernadette': 9,
 'Prophecy': 10,
 'Nashville Parthenon': 11,
 'Oh God': 12,
 'One Last Time': 13,
 'Astair': 14,
 "Let's Get It Started": 15,
 'I Never Came': 16,
 "I'm Cool": 17,
 'Folding Stars': 18,
 'Speed of Sound': 19,
 'Kladfvgbung Micshk': 20,
 'These Days': 21,
 'You Are The Only One I Love': 22,
 'We Shall All Bleed': 23,
 "Hips Don't Lie": 24,
 'Growler': 25,
 'Spite': 26,
 'Come On You Slags': 27,
 'Bitter Sweet Symphony': 28,
 'Enter Vril-Ya': 29,
 'Danger: Wildman': 30,
 'Stop Coming to My House': 31,
 'We Killed It': 32,
 'Snowball': 33,
 'All Rise': 34,
 'Bricks': 35,
 'Iridium': 36,
 'Blackmail The Universe': 37,
 'Shark': 38,
 'Metallic Rain': 39,
 'Amber Changing': 40,
 'Shadows That Move': 41,
 'It Hurts Me Too': 42,
 "Teacher's Pet": 43,
 'English Summer Rain': 44,

In [113]:


class Recommender: 

    '''
    Initializes the recommendation model
    metric: the distance metric we use (cosine)
    algorithm: algorithm used to compute the neasest neighbors (brute)
    n_neighbors: number of neighbors to use for queries (20) (very important)
    '''
    def __init__(self, metric, algorithm, k, data, decode_id_song):
        self.metric = metric
        self.algorithm = algorithm
        self.k = k
        self.data = data
        self.decode_id_song = decode_id_song
        self.model = self._recommender().fit(data)


    # initializes and fits the knn model
    def _recommender(self):
        return NearestNeighbors(metric=self.metric, algorithm=self.algorithm, 
                            n_neighbors=self.k, n_jobs=-1)





    # gets the _get_recommendations reccomendation and returns the song title with _map_indeces_to_song_title
    def make_recommendation(self, new_song, n_recommendations):
            recommendations = self._get_recommendations(new_song=new_song, n_recommendations=n_recommendations)
            return self._map_indeces_to_song_title(recommendation_ids=[idx for idx, _ in recommendations])


    # gets inputs the song, gets the kneighbors from that song
    # Also allows us to see how
    # def _get_recommendations(self, new_song, n_recommendations):
    #     recom_song_id = self._fuzzy_matching(song=new_song)
    #     # Return the n neighbors for the song id
    #     distances, indices = self.model.kneighbors(self.data[recom_song_id], 
    #                                             n_neighbors=n_recommendations+1)
    #     return sorted(list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())), 
    #                 key=lambda x: x[1])[:0:-1]
    
    def _get_recommendations(self, new_song, n_recommendations):
        recom_song_id = self._fuzzy_matching(song_artist_string=new_song)
        
        distances, indices = self.model.kneighbors(
            self.data[recom_song_id], 
            n_neighbors=n_recommendations+1
        )
        
        return sorted(
            list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())), 
            key=lambda x: x[1]
        )[:0:-1]

    # mapps the index of the song to the song title
    def _map_indeces_to_song_title(self, recommendation_ids):
        # reverse_map = {v: k for k, v in self.decode_id_song.items()}
        # return [reverse_map[i] for i in recommendation_ids]
        return [
            f"{self.decode_id_artists[i][0]} - {self.decode_id_artists[i][1]}"
            for i in recommendation_ids
        ]


    # uses Levenshtein distance to make sure that we are not making
    # any spelling mistakes with song titles
    def _fuzzy_matching(self, song_artist_string):
        import re

        def clean(s):
            return re.sub(r'[^a-z0-9 ]', '', s.lower())

        song_clean = clean(song_artist_string)

        scores = []
        for idx, title_artist in self.decode_id_title_artist.items():
            ta_clean = clean(title_artist)
            ratio = fuzz.ratio(ta_clean, song_clean)
            scores.append((idx, title_artist, ratio))

        # sort descending by score
        scores.sort(key=lambda x: x[2], reverse=True)

        best_idx, best_title_artist, best_score = scores[0]

        if best_score < 40:
            print(f"Warning: weak match for '{song_artist_string}' → '{best_title_artist}' ({best_score})")
        else:
            print(f"Good match for '{song_artist_string}' → '{best_title_artist}' ({best_score})")


        return best_idx






In [146]:
# Instantiate and fit the model
model = Recommender(metric='cosine', algorithm='brute', k=20, data=mat_songs_features, 
                    decode_id_song=decode_id_song)

model.decode_id_artists = decode_id_artists
model.decode_id_title_artist = decode_id_title_artist

song = 'Africa'
artist = 'Toto'

input_string = f"{song} - {artist}"


# code to double check artist name


new_recommendations = model.make_recommendation(new_song=input_string, n_recommendations=10)

print(f"Recommendations for {song} by {artist}:")
print("\n".join(new_recommendations))
   

Good match for 'Africa - Toto' → 'Oh Africa - Akon' (67)
Recommendations for Africa by Toto:
Common Knowledge - Quantic
Runnin' Down a Dream - Tom Petty and The Heartbreakers
Ride Me High - J.J. Cale
Bass Baby - Mr. Scruff
Square 1 - Paul Kalkbrenner
Absynthe - Paul Kalkbrenner
Left (feat. Fabian Reichelt) - Marek Hemmann
Gangsta's Paradise - Coolio
Bedtime Stories - Amon Tobin
People Get Lost - Ohmna


In [ ]:
# setup functions for spotify output
def get_top_spotify_tracks() :
    # get credentials from the json file
    cred_file = "credentials.json"
    with open(cred_file, 'r') as f:
        data = json.load(f)

    sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
        client_id=data["CLIENT_ID"],
        client_secret=data["CLIENT_SECRET"],
        redirect_uri=data["REDIRECT_URI"],
        scope=data["SCOPE"]
    ))

    top_tracks = sp.current_user_top_tracks(time_range='long_term', limit=20)
    return top_tracks





In [122]:
# NEW OUTPUT: different format, spit out the artists
# LASTFM API
# Goal: grab the users top songs

fuzzy_val = 60 # logic to lower fuzzy value if IndexError



class Recommender: 

    '''
    Initializes the recommendation model
    metric: the distance metric we use (cosine)
    algorithm: algorithm used to compute the neasest neighbors (brute)
    n_neighbors: number of neighbors to use for queries (20) (very important)
    '''
    def __init__(self, metric, algorithm, k, data, decode_id_song):
        self.metric = metric
        self.algorithm = algorithm
        self.k = k
        self.data = data
        self.decode_id_song = decode_id_song
        self.model = self._recommender().fit(data)


    # initializes and fits the knn model
    def _recommender(self):
        return NearestNeighbors(metric=self.metric, algorithm=self.algorithm, 
                            n_neighbors=self.k, n_jobs=-1)





    # gets the _get_recommendations reccomendation and returns the song title with _map_indeces_to_song_title
    def make_recommendation(self, new_song, n_recommendations):
            recommendations = self._get_recommendations(new_song=new_song, n_recommendations=n_recommendations)
            return self._map_indeces_to_song_title(recommendation_ids=[idx for idx, _ in recommendations])


    # gets inputs the song, gets the kneighbors from that song
    # Also allows us to see how
    # def _get_recommendations(self, new_song, n_recommendations):
    #     recom_song_id = self._fuzzy_matching(song=new_song)
    #     # Return the n neighbors for the song id
    #     distances, indices = self.model.kneighbors(self.data[recom_song_id], 
    #                                             n_neighbors=n_recommendations+1)
    #     return sorted(list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())), 
    #                 key=lambda x: x[1])[:0:-1]
    
    def _get_recommendations(self, new_song, n_recommendations):
        recom_song_id = self._fuzzy_matching(song_artist_string=new_song)
        
        distances, indices = self.model.kneighbors(
            self.data[recom_song_id], 
            n_neighbors=n_recommendations+1
        )
        
        return sorted(
            list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())), 
            key=lambda x: x[1]
        )[:0:-1]

    # mapps the index of the song to the song title
    def _map_indeces_to_song_title(self, recommendation_ids):
        # reverse_map = {v: k for k, v in self.decode_id_song.items()}
        # return [reverse_map[i] for i in recommendation_ids]
        return [
            f"{self.decode_id_artists[i][0]} - {self.decode_id_artists[i][1]}"
            for i in recommendation_ids
        ]


    # uses Levenshtein distance to make sure that we are not making
    # any spelling mistakes with song titles
    # TODO update to new fuzzymatch
    # def _fuzzy_matching(self, song):
    #     match_tuple = []
    #     # get match
    #     for title, idx in self.decode_id_song.items():
    #         ratio = fuzz.ratio(title.lower(), song.lower())
    #         if ratio >= fuzzy_val:
    #             match_tuple.append((title, idx, ratio))
                
    #     match_tuple = sorted(match_tuple, key=lambda x: x[2])[::-1]
        
    #     return match_tuple[0][1]

    def _fuzzy_matching(self, song_artist_string):
        import re

        def clean(s):
            return re.sub(r'[^a-z0-9 ]', '', s.lower())

        song_clean = clean(song_artist_string)

        scores = []
        for idx, title_artist in self.decode_id_title_artist.items():
            ta_clean = clean(title_artist)
            ratio = fuzz.ratio(ta_clean, song_clean)
            scores.append((idx, title_artist, ratio))

        # sort descending by score
        scores.sort(key=lambda x: x[2], reverse=True)

        best_idx, best_title_artist, best_score = scores[0]

        if best_score < 40:
            print(f"Warning: weak match for '{song_artist_string}' → '{best_title_artist}' ({best_score})")
        else:
            print(f"Good match for '{song_artist_string}' → '{best_title_artist}' ({best_score})")


        return best_idx





   



top_tracks = get_top_spotify_tracks()

track_ids = [track['id'] for track in top_tracks['items']]

response = requests.get(
        "https://api.reccobeats.com/v1/audio-features",
        params={"ids": ",".join(track_ids)},
        headers={"Accept": "application/json"}
    )

features = response.json().get("content", [])

for i, (track, feat) in enumerate(zip(top_tracks['items'], features)):
    artist_name = track['artists'][0]['name']
    track_name = track['name']

    model = Recommender(metric='cosine', algorithm='brute', k=20, data=mat_songs_features, 
                    decode_id_song=decode_id_song)

    model.decode_id_artists = decode_id_artists
    model.decode_id_title_artist = decode_id_title_artist


    loop_complete = False
    while not loop_complete:
        try:
            new_recommendations = model.make_recommendation(new_song=track_name, n_recommendations=10)
            
            
            print(f"Recommendations for {track_name} by {artist_name} are:")
            print("\n".join(new_recommendations))

            loop_complete = True
            fuzzy_val = 60

        except IndexError:
            fuzzy_val -= 5



Good match for 'Can't Hardly Wait - 2008 Remaster' → 'Can't Hardly Stand It - Charlie Feathers' (67)
Recommendations for Can't Hardly Wait - 2008 Remaster by The Replacements are:
Sun in an Empty Room - The Weakerthans
Not Ready To Love - Rufus Wainwright
A Moment Like This - Leona Lewis
Redneck Yacht Club - Craig Morgan
Grand Designs - Rush
Black Heart Romance - My Dying Bride
Me Or The Papes - Jeru the Damaja
I Miss My Friend - Darryl Worley
Mom & Dad - Frank Zappa
Vulnerable - Secondhand Serenade
Good match for 'Cemetry Gates - 2011 Remaster' → 'Cemetry Gates - The Smiths' (68)
Recommendations for Cemetry Gates - 2011 Remaster by The Smiths are:
Hand in Glove - The Smiths
Ghost Under Rocks - Ra Ra Riot
Bigmouth Strikes Again - The Smiths
Olympic Airways - Foals
St. Peter's Day Festival - Ra Ra Riot
Spanish Sahara - Foals
Southbound - The Allman Brothers Band
Bell - Swirlies
Second Sense - Jon Hopkins
Vicar in a Tutu - The Smiths
Good match for 'A Change Of Heart' → 'Last Chance - Je